# Module 2 · Lesson 01: Zero-Shot Prompting

**Zero-shot prompting** means asking the model to perform a task *without any examples*.
The model relies entirely on its training data and your instructions.

## What you will learn
1. Effective zero-shot prompt patterns
2. Output **format specification**
3. **Role/persona** assignment
4. Zero-shot **classification**
5. **Structured output** (JSON) extraction
6. Using **constraints** to control output

In [ ]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
def ask(prompt, system=None, temperature=0.7, max_tokens=1200):
    """Helper: call GPT and return text."""
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=msgs,
        temperature=temperature, 
        max_tokens=max_tokens
    )
    return r.choices[0].message.content
 
if client:
    print("Client is ready")
else:
    print("Check API key and OpenAI client")
 

Client is ready


---
## 1. Simple Zero-Shot

The simplest form — just ask directly:

In [2]:
result = ask("What is the capital of France?")
display(Markdown(f"**Q**: What is the capital of France\n\n**A**: {result}"))

**Q**: What is the capital of France

**A**: The capital of France is Paris.

---
## 2. Format Specification

Tell the model **exactly** what format you want:

In [4]:
prompt = """Extract the email address from this text and return ONLY the email, nothing else:

Hi, you can reach me at john.doe@example.com for more information.
"""

result = ask(prompt, max_tokens=50)
display(Markdown(f"**Extracted:** `{result.strip()}`"))

**Extracted:** `john.doe@example.com`

---
## 3. Role / Persona Assignment

The system prompt sets *who* the model should be:

In [5]:
result = ask(
    prompt="Explain what an API is to a non-technical person.",
    system="You are a professional technical writer who explains complex concepts simply."
)

display(Markdown(f"### Technical Writer\n\n {result}"))

### Technical Writer

 Sure! Think of an API, or Application Programming Interface, as a waiter in a restaurant. When you go to a restaurant, you have a menu that lists the dishes you can order. You tell the waiter what you want, and the waiter communicates your order to the kitchen. Once your food is ready, the waiter brings it back to you.

In this analogy, the restaurant is like a software application, the menu is the API documentation that describes what you can request, and the waiter is the API itself. It takes your requests, sends them to the right place, and brings back the information or services you need.

So, an API allows different software programs to communicate with each other, enabling them to share data and functionality. This makes it easier for developers to build applications without having to understand all the inner workings of other software.

---
## 4. Zero-Shot Classification

LLMs are excellent **zero-shot classifiers**. No training data needed!

In [6]:
texts = [
    "I absolutely love this product! Best purchase ever",
    "The shipping was delayed and the item arrived damaged",
    "It's okay, nothing special but does the job"
]

print(f"{'Text':<55} {'Sentiment'}")
print("_" * 101)

for text in texts:
    prompt = f"Classify the sentiment as exactly one word: positive, negative or neutral.\n\nText:{text}\n\nClassification:"
    sentiment = ask(prompt, temperature=0, max_tokens=10).strip().lower()
    emoji = {"positive": "🟢", "negative": "🔴", "neutral": "🟡"}.get(sentiment, "⚪")
    print(f"{text[:52]+'...':<55} {emoji} {sentiment}")

Text                                                    Sentiment
_____________________________________________________________________________________________________
I absolutely love this product! Best purchase ever...   🟢 positive
The shipping was delayed and the item arrived damage... 🔴 negative
It's okay, nothing special but does the job...          🟡 neutral


# Role - Context - Structure


In [8]:
result = ask(
    prompt="Explain what an API is to a 10 years old child. Do it in 3 short sentences. Put them in ordered bullets.",
    system="You are my school teacher who loves analogies and explains everything in an easy way"
)

display(Markdown(f"### Technical Writer:\n\n{result}"))

### Technical Writer:

Sure! Here’s a simple way to understand an API:

1. Imagine an API like a waiter in a restaurant who takes your order and brings you your food.  
2. When you want something from a computer program, the API is the waiter that helps you ask for it.  
3. Just like the waiter knows how to get what you want from the kitchen, the API knows how to get information from different programs!  

> 💡 Use `temperature=0` for classification to get deterministic, consistent results.

---
## 5. Structured Output (JSON)

In [10]:
import json

prompt = """Extract information from this text and return as valid JSON:

Text: Meeting with Sarah Johnson scheduled for March 15, 2026 at 2:30PM to discuss the Q1 budget report.

Return JSON fields: attendee, date, time, topic

DO NOT wrap the output in markdown code fences or any other formating.
Return ONLY the raw JSON object, nothing else.

JSON:
"""

result = ask(prompt, temperature=0)
try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Invalid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [11]:
response_json = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role":"user", "content": prompt}],
    response_format={"type":"json_object"}, # force valid JSON output
    temperature=0
)

new_res = response_json.choices[0].message.content

try:
    parsed = json.loads(new_res)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {new_res}")
    print("Invalid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [14]:
import json

response_json = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role":"user", "content": "Return a short greeting and a lucky number"}],
    response_format={"type":"json_schema",
                     "json_schema":{
                         "name": "greeting_response",
                         "schema":{
                             "type":"object",
                             "properties": {
                                 "greeting": {"type":"string"},
                                 "luckyNumber": {"type":"integer"}
                             },
                             "required":["greeting", "luckyNumber"],
                             "additionalProperties": False
                         },
                         "strict":True
                     }
    }, 
    temperature=0
)

new_res = response_json.choices[0].message.content
parsed = json.loads(new_res)

print(json.dumps(parsed, indent=2))

{
  "greeting": "Hello! Wishing you a wonderful day!",
  "luckyNumber": 7
}


In [16]:
import json
 
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Describe an API endpoint for user login."}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "api_endpoint",
            "schema": {
                "type": "object",
                "properties": {
                    "endpoint": {"type": "string"},
                    "method": {
                        "type": "string",
                        "enum": ["GET", "POST", "PUT", "DELETE"]
                    },
                    "request_body": {
                        "type": "object",
                        "properties": {
                            "email": {"type": "string"},
                            "password": {"type": "string"}
                        },
                        "required": ["email", "password"],
                        "additionalProperties": False
                    },
                    "response": {
                        "type": "object",
                        "properties": {
                            "token": {"type": "string"},
                            "expires_in": {"type": "integer"}
                        },
                        "required": ["token", "expires_in"],
                        "additionalProperties": False
                    }
                },
                "required": ["endpoint", "method", "request_body", "response"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    temperature=0
)
 
parsed = json.loads(response.choices[0].message.content)
print(json.dumps(parsed, indent=2))

{
  "endpoint": "/api/login",
  "method": "POST",
  "request_body": {
    "email": "user@example.com",
    "password": "userpassword"
  },
  "response": {
    "token": "abc123xyz",
    "expires_in": 3600
  }
}


---
## 6. Constraints

Adding explicit **constraints** controls length, format, and content:

In [18]:
prompt = """Write a product description for a wireless mouse.
Constraints:
- Maximum 50 words
- Include at least one benefit
- Do not mention price
- End with a call to action

Description:
"""

result = ask(prompt)
word_count = len(result.split())

display(Markdown(f"**Result**: {result}\n\n **Word count**: {word_count}"))

**Result**: Experience seamless navigation with our sleek wireless mouse, designed for ultimate comfort and precision. Enjoy the freedom of movement without tangled cords, enhancing your productivity whether at home or on the go. Elevate your workspace today—grab yours and transform the way you work!

 **Word count**: 43

# Multi-dimensional contraints (style - structure - semantics)

In [21]:
prompt = """
Write a product description for wireless mouse.

Contraints:
- More than 40 words less than 60
- Exactly 2 sentences
- First sentece: describe features
- Second sentece: emphasize a user benefit
- Include exactly 1 emoji
- Must contain the word "precision"
- Do NOT use passive voice
- End with a call to action

Return ONLY the description
"""

result = ask(prompt)

word_count = len(result.split())

display(Markdown(f"**Result**: {result}\n\n **Word count**: {word_count}"))


**Result**: Experience seamless navigation with our wireless mouse, featuring advanced optical sensors for precision tracking, customizable buttons, and an ergonomic design for ultimate comfort. Boost your productivity and enjoy a clutter-free workspace—grab yours today! 🖱️

 **Word count**: 34

---
## 7. Prompt Gallery: Real-World System Prompts

Let's study system prompts from **real community projects**. Each uses a different technique
to get reliable, high-quality outputs.

| Source | Key Technique |
|--------|---------------|
| Gmail Summarizer | Structured rules + labels |
| Budget Travel Agent | Role + constraint + format |
| Biomedical Summariser | Audience awareness |
| Code Explainer | Section structure |

In [22]:
# ── Prompt Gallery: 4 real-world system prompts ─────────
 
gallery = {
    "Gmail Summarizer": {
        "prompt": """You summarize email threads. For each email:
- Subject line (max 10 words)
- Label: ACTION_REQUIRED | FYI | PROMO | URGENT
- Summary (max 2 sentences)
- Has link: yes/no
Return as a numbered list.""",
        "technique": "Structured rules with labels and constraints",
    },
    "Budget Travel Agent": {
        "prompt": """You are a budget travel advisor. For any destination:
1. List top 5 FREE attractions
2. Suggest 3 budget restaurants (under $15/meal)
3. Give one money-saving local tip
Respond in markdown with headers.""",
        "technique": "Role + numbered constraints + format (markdown)",
    },
    "Biomedical Summariser": {
        "prompt": """Summarize biomedical research articles for a mixed audience:
students, early researchers, and professionals.
- Use bullet points for key findings
- Highlight methodology and sample size
- Note limitations and future directions
Tone: professional, clear, accessible.""",
        "technique": "Audience awareness + structure + tone",
    },
    "Code Explainer": {
        "prompt": """You explain code to developers. Structure your response as:
1) Direct Answer (1-2 sentences)
2) Explanation (why it works)
3) Example (working code snippet)
4) Common Pitfalls (what to avoid)
5) Next Steps (what to learn next)""",
        "technique": "Section-structured output format",
    }
}
 
# Display each prompt with analysis
for name, info in gallery.items():
    print(f"\n{'=' * 60}")
    print(f"  {name}")
    print(f"  Technique: {info['technique']}")
    print(f"{'=' * 60}")
    print(info['prompt'])


  Gmail Summarizer
  Technique: Structured rules with labels and constraints
You summarize email threads. For each email:
- Subject line (max 10 words)
- Label: ACTION_REQUIRED | FYI | PROMO | URGENT
- Summary (max 2 sentences)
- Has link: yes/no
Return as a numbered list.

  Budget Travel Agent
  Technique: Role + numbered constraints + format (markdown)
You are a budget travel advisor. For any destination:
1. List top 5 FREE attractions
2. Suggest 3 budget restaurants (under $15/meal)
3. Give one money-saving local tip
Respond in markdown with headers.

  Biomedical Summariser
  Technique: Audience awareness + structure + tone
Summarize biomedical research articles for a mixed audience:
students, early researchers, and professionals.
- Use bullet points for key findings
- Highlight methodology and sample size
- Note limitations and future directions
Tone: professional, clear, accessible.

  Code Explainer
  Technique: Section-structured output format
You explain code to developers. 

In [23]:
travel_result = ask(
    prompt="I'm visiting Lisbon, Portugal for 3 days on a tight budget.",
    system=gallery["Budget Travel Agent"]["prompt"]
)

display(Markdown(f"### Budget Travel Agent Response\n\n{travel_result}"))

### Budget Travel Agent Response

# Budget Travel Guide to Lisbon, Portugal

## Top 5 FREE Attractions
1. **Alfama District**: Wander through the narrow streets of Lisbon's oldest neighborhood, filled with charming buildings and stunning views.
2. **Miradouro de Santa Catarina**: Enjoy panoramic views of the city and the Tagus River from this popular viewpoint.
3. **Praça do Comércio**: Stroll around this grand square by the river, often considered one of the most iconic places in Lisbon.
4. **Lisbon Cathedral (Sé de Lisboa)**: Visit this beautiful cathedral and explore its surroundings without any entry fee.
5. **Jardim da Estrela**: Relax in this peaceful garden, perfect for a picnic or a leisurely walk among the greenery.

## Budget Restaurants (Under $15/meal)
1. **Time Out Market**: Although individual stalls may vary in price, you can find delicious options like sandwiches and local tapas within your budget.
2. **O Prego da Peixaria**: Try their famous fish prego (a grilled fish sandwich) that’s both delicious and affordable.
3. **A Cevicheria**: This spot offers budget-friendly ceviche and other seafood dishes, usually around the $15 mark.

## Money-Saving Local Tip
**Use the Tram 28**: Instead of paying for a guided tour, take the iconic Tram 28. It winds through many of the city's most famous neighborhoods and attractions, offering a scenic and budget-friendly way to explore Lisbon's highlights. Just make sure to validate your ticket!

In [ ]:
gmail_result = ask(
    """Here's an email thread:
From: marketing@company.com
Subject: Re: Q2 Campaign Launch — Asset Review Needed
Body: Hi team, please review the attached creatives for the Q2 social campaign.
We need approvals by Friday. The campaign landing page is at https://company.com/q2launch.
Let me know if any changes are needed. Thanks!""",
    system=gallery["Gmail Summarizer"]["prompt"]
)
display(Markdown(f"### Gmail Summarizer Response\n\n{gmail_result}"))
print("\nNotice the consistent labeling and structured bullet format!")


### Gmail Summarizer Response

1. Subject line: Q2 Campaign Launch — Asset Review Needed  
   Label: ACTION_REQUIRED  
   Summary: The marketing team requests a review and approval of the attached creatives for the Q2 social campaign by Friday. The campaign landing page is provided for reference.  
   Has link: yes  


Notice the consistent labeling and structured bullet format!


In [25]:
bio_result = ask(
    """A 2024 study published in The Lancet examined the efficacy of a novel mRNA-based
therapeutic vaccine for stage III melanoma. The randomized, double-blind trial enrolled
340 patients across 22 clinical sites. Results showed a 44% reduction in recurrence risk
(HR 0.56, 95% CI 0.40–0.78, p=0.0007) over a 24-month follow-up. Common adverse effects
included fatigue (32%) and injection-site reactions (28%). The authors noted the relatively
short follow-up period and homogeneous population (predominantly Caucasian, median age 58)
as key limitations, and called for larger Phase III trials with diverse cohorts.""",
    system=gallery["Biomedical Summariser"]["prompt"]
)
display(Markdown(f"### Biomedical Summariser Response\n\n{bio_result}"))
print("\nNotice how it highlights methodology, limitations, and stays accessible!")

### Biomedical Summariser Response

**Key Findings:**
- The study evaluated a novel mRNA-based therapeutic vaccine for patients with stage III melanoma.
- Results indicated a significant 44% reduction in the risk of recurrence over 24 months (Hazard Ratio [HR] 0.56, 95% Confidence Interval [CI] 0.40–0.78, p=0.0007).

**Methodology:**
- Study Design: Randomized, double-blind trial.
- Sample Size: 340 patients.
- Recruitment: Conducted across 22 clinical sites.

**Adverse Effects:**
- Common side effects included:
  - Fatigue (32% of participants)
  - Injection-site reactions (28% of participants)

**Limitations:**
- The follow-up period was relatively short (24 months).
- The study population was predominantly Caucasian with a median age of 58, suggesting a lack of diversity.

**Future Directions:**
- The authors recommend conducting larger Phase III trials to include more diverse cohorts to enhance the generalizability of the findings.


Notice how it highlights methodology, limitations, and stays accessible!


In [26]:
code_result = ask(
    """Explain this Python code:
result = {k: v for k, v in sorted(data.items(), key=lambda item: item[1], reverse=True)[:5]}""",
    system=gallery["Code Explainer"]["prompt"]
)
display(Markdown(f"### Code Explainer Response\n\n{code_result}"))
print("\nNotice the 5-section structure: Answer → Explanation → Example → Pitfalls → Next Steps!")

### Code Explainer Response

1) **Direct Answer:** This Python code creates a dictionary called `result` that contains the top five key-value pairs from the `data` dictionary, sorted by their values in descending order.

2) **Explanation:** The `sorted()` function sorts the items of the `data` dictionary based on their values (the second item in each key-value pair), as specified by the `key=lambda item: item[1]`. The `reverse=True` argument sorts the items in descending order. The slicing `[:5]` limits the results to the top five items, which are then used to construct the new dictionary using a dictionary comprehension.

3) **Example:**
   ```python
   data = {'a': 10, 'b': 40, 'c': 30, 'd': 20, 'e': 50, 'f': 60}
   result = {k: v for k, v in sorted(data.items(), key=lambda item: item[1], reverse=True)[:5]}
   print(result)  # Output: {'f': 60, 'e': 50, 'b': 40, 'c': 30, 'd': 20}
   ```

4) **Common Pitfalls:** One common pitfall is forgetting to handle cases where `data` has fewer than five items, which could lead to unexpected behavior or incomplete results. Additionally, if the values are not comparable or if they are not unique, the sorting may not yield clear top values.

5) **Next Steps:** To deepen your understanding, explore how to handle ties in values when sorting, learn about other sorting techniques in Python, and practice using the `collections` module for more efficient data structures.


Notice the 5-section structure: Answer → Explanation → Example → Pitfalls → Next Steps!


> **Exercise:** Pick a business domain you're interested in (e.g., fitness coaching,
> legal review, recipe generation) and write your own system prompt following the
> patterns above. Test it with 3 different user queries.

---
## Key Takeaways 📝

| Technique | When to Use |
|-----------|------------|
| **Direct question** | Simple factual queries |
| **Format specification** | When you need specific output format |
| **Role assignment** | To control tone, expertise, style |
| **Classification** | Categorizing text (use temp=0) |
| **JSON extraction** | Structured data from unstructured text |
| **Constraints** | Controlling length, style, content boundaries |
| **Prompt gallery** | Study real prompts to develop critical analysis skills |

### Zero-Shot Tips
1. Be **specific** about what you want
2. Specify the **output format** explicitly
3. Use **roles/personas** to guide behaviour
4. Add **negative constraints** (what NOT to do)
5. Use `temperature=0` for deterministic tasks

---
**Next:** `02_few_shot_examples.ipynb` — Improve results by providing examples